# Stochastic Goose — PARALLEL benchmark sweep (Colab, OFFLINE, Drive-persisted)

Runs the **Stochastic Goose** replication (Dries Smit / Tufa Labs, 1st place ARC-AGI-3 preview, 12.58%)
**offline** (local env_files — no API key/internet) on all 4 games, logging the action-step at each
level-clear → the **same metric as our staircase** (env-steps to clear L1/L2/L3).

- **Parallel:** each `(game, seed)` runs as its own process, `CONCURRENCY` at a time.
- **Drive-persisted:** every run writes its `result.json` (+ log) to Google Drive continuously
  (on each level-up / every N actions / every ~2 min) → a disconnect loses nothing; re-running only
  redoes unfinished `(game, seed)` jobs.

> **Set Runtime ▸ GPU.** Goose's per-step CNN is the bottleneck; GPU helps.

## 1. Setup (clone + install; OFFLINE needs no API key)

In [ ]:
import os, sys
REPO_URL = "https://github.com/LavetteSinsora/ProjectArceus.git"; REPO = "/content/ProjectArceus"
if not os.path.isdir(REPO):
    !git clone --depth 1 $REPO_URL $REPO
%cd /content/ProjectArceus
!git pull --ff-only -q || true
!pip -q install "arc-agi>=0.9.8" "arcengine>=0.9.3" torch
import torch; print("CUDA:", torch.cuda.is_available())

## 2. Mount Drive + config

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/goose_results"; os.makedirs(OUT, exist_ok=True)
RUNNER = "/content/ProjectArceus/replication/card_stochastic_goose/goose_offline_run.py"

GAMES       = ["ls20", "tu93", "re86", "g50t"]
SEEDS       = [0, 1]
MAX_ACTIONS = {"ls20": 200_000, "tu93": 600_000, "re86": 1_000_000, "g50t": 300_000}
MAX_MINUTES = 120        # wall-clock cap per (game, seed)
CONCURRENCY = 3          # parallel processes (Colab ~2 vCPU + 1 GPU → 2–3 is the sweet spot)
print("results →", OUT, "| games", GAMES, "| seeds", SEEDS, "| concurrency", CONCURRENCY)

## 3. Run the sweep in parallel (Drive-persisted)

Launches all `len(GAMES)×len(SEEDS)` jobs, `CONCURRENCY` at a time. Each is a separate process running
`goose_offline_run.py`, writing `goose_<game>_seed<seed>.json` + `log_<game>_seed<seed>.txt` to Drive
incrementally. Re-runnable.

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor

jobs = [(g, s) for g in GAMES for s in SEEDS]
def launch(job):
    g, s = job
    cmd = [sys.executable, RUNNER, "--game", g, "--seed", str(s), "--out", OUT,
           "--max-actions", str(MAX_ACTIONS[g]), "--max-minutes", str(MAX_MINUTES)]
    with open(f"{OUT}/log_{g}_seed{s}.txt", "w") as f:
        rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT).returncode
    print(f"[{'ok ' if rc == 0 else 'FAIL'}] {g} seed{s}")
    return rc

print(f"launching {len(jobs)} runs, {CONCURRENCY} at a time → {OUT}")
with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
    list(ex.map(launch, jobs))
print("ALL DONE. results + logs in", OUT)

## 4. Aggregate — steps-to-clear-each-level (reads from Drive; safe to run anytime)

In [ ]:
import glob, json
import numpy as np
from collections import defaultdict
rows = [json.load(open(f)) for f in glob.glob(f"{OUT}/goose_*.json")]
agg = defaultdict(lambda: defaultdict(list))
for r in rows:
    for L, st in r.get("level_steps", {}).items():
        agg[r["game"]][int(L)].append(st)
print(f"{'game':>5} | steps to clear L1 / L2 / L3 (median over seeds; — = not cleared)  [n runs]")
for g in GAMES:
    cells = []
    for L in (1, 2, 3):
        v = agg[g].get(L, [])
        cells.append(f"{np.median(v):,.0f}" if v else "—")
    nr = sum(1 for r in rows if r["game"] == g)
    print(f"{g:>5} | {cells[0]:>10} {cells[1]:>10} {cells[2]:>10}   [{nr}]")
print("\n(For our methods, compare to the staircase in exp_014_figures_and_results.)")